# 40 — Canonical trees (Group A)

One simulated recovery study per canonical tree, probit and logit kept as separate analyses you compare yourself. Data is simulated first, then handed to the package through `ingest.load_data` exactly as a real response file would be.

## A1. Dichotomous disengagement, Rasch

## Setup — install the package, load the harness simulator

In [ ]:
import sys
!{sys.executable} -m pip install -e "C:\Users\mehoujm\OneDrive - James Madison University\Spring 26 classes & projects\Practicum Python IRtree package\PACKAGE\Irtree"

In [ ]:
import sys, warnings, pathlib, importlib.util
import numpy as np, pandas as pd
np.set_printoptions(precision=4, suppress=True)
warnings.filterwarnings('ignore')

import irtree.tree_spec   as ts
import irtree.node_models as nm
import irtree.priors      as pr
import irtree.constraints as con
import irtree.ingest      as ing
import irtree.fit         as ft

# The data-generating code (simulate_irtree) lives in the test harness, NOT in
# the package: simulation is deliberately kept separate from fitting.  Load it
# the same way notebooks 40 & 50 do.
ROOT = pathlib.Path.cwd().parents[1]
_hspec = importlib.util.spec_from_file_location("_harness", ROOT / "tests" / "_harness.py")
h = importlib.util.module_from_spec(_hspec); _hspec.loader.exec_module(h)

# probit -> logit conversion, per parameter role (TESTING.md 9.3), for your
# own hand comparisons between the two fits.
D_PL = 1.702
def to_logit(value, role):
    if role == 'slope':        return D_PL * np.asarray(value)
    if role in ('location','correlation','difficulty'): return np.asarray(value)
    raise ValueError(role)

## Study size

In [ ]:
# ===================== study size =====================
# One simulated dataset per scenario. Evidence-mode warmup is the package default, so any fit with an MH block
# (logit nodes, 3PL guessing, an estimated correlation) needs N_ITER >= ~1600;
# pure-Gibbs probit Rasch has no such floor.
P, I     = 1500, 30
N_ITER   = 4000
N_CHAINS = 1
SEED     = 20250413
print(f'P={P}  I={I}  N_ITER={N_ITER}  N_CHAINS={N_CHAINS}  SEED={SEED}')

## Two small helpers for this notebook
`responses_frame` makes the simulated matrix look like a real response file for `ingest`; `compare` prints posterior vs truth. Everything else is called explicitly so you can rehearse the API.

In [ ]:
def responses_frame(Y):
    """Turn a simulated [P, I] response matrix into the kind of keyed table a
    real study would hand to ingest.load_data: one row per person, item ids as
    column headers, an explicit person_id. (load_data also reads a CSV path.)"""
    Y = np.asarray(Y); Pn, In = Y.shape
    df = pd.DataFrame(Y, columns=[f'item{i+1}' for i in range(In)])
    df.insert(0, 'person_id', [f'p{p+1}' for p in range(Pn)])
    return df

def person_frame(X, names):
    df = pd.DataFrame(np.asarray(X), columns=list(names))
    df.insert(0, 'person_id', [f'p{p+1}' for p in range(np.asarray(X).shape[0])])
    return df

def item_frame(Q, names):
    df = pd.DataFrame(np.asarray(Q), columns=list(names))
    df.insert(0, 'item_id', [f'item{i+1}' for i in range(np.asarray(Q).shape[0])])
    return df

def compare(true_vec, prefix, fit, label=''):
    """Posterior mean + 95% interval for every parameter named `prefix*`,
    next to the known truth. Rehearses pulling estimates out of fit.results.
    Returns the DataFrame so you can compare however you like."""
    names = fit.results[0].param_names
    cols  = [j for j, n in enumerate(names) if n.startswith(prefix)]
    draws = np.concatenate([np.asarray(r.chain, float) for r in fit.results])[:, cols]
    tbl = pd.DataFrame({
        'param'    : [names[j] for j in cols],
        'true'     : np.asarray(true_vec, float).ravel()[:len(cols)],
        'post_mean': draws.mean(0),
        'ci_2.5'   : np.percentile(draws, 2.5, axis=0),
        'ci_97.5'  : np.percentile(draws, 97.5, axis=0),
    })
    tbl['abs_err'] = (tbl['post_mean'] - tbl['true']).abs()
    if label: print(label)
    print(tbl.to_string(index=False,
          formatters={c: '{:+.3f}'.format for c in ['true','post_mean','ci_2.5','ci_97.5','abs_err']}))
    return tbl

**Simulate** (data-generating step — separate from the package)

In [ ]:
spec_dis = ts.disengagement_tree(binary=True)
rng = np.random.default_rng(SEED)
theta1  = rng.normal(0, 1, P)        # node 1: rapid-guessing propensity
theta2  = rng.normal(0, 1, P)        # node 2: ability
b1_true = rng.normal(0.6, 0.7, I)    # node-1 difficulties
b2_true = rng.normal(0.0, 1.0, I)    # node-2 difficulties


truth = [dict(theta=theta1, b=b1_true), dict(theta=theta2, b=b2_true)]
Y_probit = h.simulate_irtree(spec_dis, truth, P, I, np.random.default_rng(SEED), node_links=['probit','probit'])
Y_logit  = h.simulate_irtree(spec_dis, truth, P, I, np.random.default_rng(SEED), node_links=['logit','logit'])
resp_probit, resp_logit = responses_frame(Y_probit), responses_frame(Y_logit)
print('node-1 category counts (probit):', np.bincount(Y_probit.ravel())[1:])

In [ ]:
resp_probit

**Probit** — ingest, node models, fit, compare

In [ ]:
data = ing.load_data(resp_probit, spec=spec_dis, categories_start=1) # load in the data


node_models = [nm.ProbitRaschNode(),   # node 1: rapid-guessing (Rasch, probit)
               nm.ProbitRaschNode()]   # node 2: ability


fit = ft.irtree_fit(data, spec_dis, node_models, n_iter=3000, n_chains=1, seed=SEED)


In [ ]:
fit.summary(max_params=8)
fit.diagnostics()
compare(b1_true, 'b_1_', fit, 'node-1 difficulties (probit)')
compare(b2_true, 'b_2_', fit, 'node-2 difficulties (probit)')

**Logit** — the same study, logit link

In [ ]:
data = ing.load_data(resp_logit, spec=spec_dis, categories_start=1)


node_models = [nm.LogitRaschNode(), nm.LogitRaschNode()]   # both nodes logit (MH)


fit = ft.irtree_fit(data, spec_dis, node_models, n_iter=15000, n_chains=N_CHAINS, seed=SEED, warmup_mode  = "fixed", n_warmup = 10000)




In [ ]:
# logit nodes are MH; evidence-mode warmup reports where each block froze:
print('accept rates:', {k: round(v,3) for k,v in fit.results[0].accept_rates.items()})
print('froze at    :', fit.results[0].frozen_at)

In [ ]:
compare(b1_true, 'b_1_', fit, 'node-1 difficulties (logit)')
compare(b2_true, 'b_2_', fit, 'node-2 difficulties (logit)')

## A2. Distractor tree, Rasch (nested)

**Simulate** (data-generating step — separate from the package)

In [ ]:
spec_dist = ts.distractor_tree(); R = spec_dist.n_nodes
rng = np.random.default_rng(SEED+1)
b_true = [rng.normal(0, 1, I) for _ in range(R)]
theta  = [rng.normal(0, 1, P) for _ in range(R)]
truth  = [dict(theta=theta[r], b=b_true[r]) for r in range(R)]
Y_probit = h.simulate_irtree(spec_dist, truth, P, I, np.random.default_rng(SEED+1), node_links=['probit']*R)
Y_logit  = h.simulate_irtree(spec_dist, truth, P, I, np.random.default_rng(SEED+1), node_links=['logit']*R)
resp_probit, resp_logit = responses_frame(Y_probit), responses_frame(Y_logit)
print('distractor tree: R =', R, 'nodes; categories present', np.unique(Y_probit))

**Probit** — ingest, node models, fit, compare

In [ ]:
data = ing.load_data(resp_probit, spec=spec_dist, categories_start=1)
node_models = [nm.ProbitRaschNode() for _ in range(R)]
fit = ft.irtree_fit(data, spec_dist, node_models, n_iter=N_ITER, n_chains=N_CHAINS, seed=SEED+1)
for r in range(R):
    compare(b_true[r], f'b_{r+1}_', fit, f'node-{r+1} difficulties (probit)')

**Logit** — the same study, logit link

In [ ]:
data = ing.load_data(resp_logit, spec=spec_dist, categories_start=1)
node_models = [nm.LogitRaschNode() for _ in range(R)]
fit = ft.irtree_fit(data, spec_dist, node_models, n_iter=N_ITER, n_chains=N_CHAINS, seed=SEED+1)
for r in range(R):
    compare(b_true[r], f'b_{r+1}_', fit, f'node-{r+1} difficulties (logit)')

## A3. ERS tree, Rasch (shared θ across two nodes)

**Simulate** (data-generating step — separate from the package)

In [ ]:
spec_ers = ts.ers_tree(); R = spec_ers.n_nodes
print('dim_to_nodes:', spec_ers.dim_to_nodes)
rng = np.random.default_rng(SEED+2)
theta0, theta_sh = rng.normal(0,1,P), rng.normal(0,1,P)   # nodes 2 & 3 share theta_sh
thetas = [theta0, theta_sh, theta_sh]
b_true = [rng.normal(0,1,I) for _ in range(R)]
truth  = [dict(theta=thetas[r], b=b_true[r]) for r in range(R)]
Y_probit = h.simulate_irtree(spec_ers, truth, P, I, np.random.default_rng(SEED+2), node_links=['probit']*R)
Y_logit  = h.simulate_irtree(spec_ers, truth, P, I, np.random.default_rng(SEED+2), node_links=['logit']*R)
resp_probit, resp_logit = responses_frame(Y_probit), responses_frame(Y_logit)

**Probit** — ingest, node models, fit, compare

In [ ]:
data = ing.load_data(resp_probit, spec=spec_ers, categories_start=1)
node_models = [nm.ProbitRaschNode() for _ in range(R)]
fit = ft.irtree_fit(data, spec_ers, node_models, n_iter=N_ITER, n_chains=N_CHAINS, seed=SEED+2)
for r in range(R):
    compare(b_true[r], f'b_{r+1}_', fit, f'node-{r+1} difficulties (probit)')

**Logit** — the same study, logit link

In [ ]:
data = ing.load_data(resp_logit, spec=spec_ers, categories_start=1)
node_models = [nm.LogitRaschNode() for _ in range(R)]
fit = ft.irtree_fit(data, spec_ers, node_models, n_iter=N_ITER, n_chains=N_CHAINS, seed=SEED+2)
for r in range(R):
    compare(b_true[r], f'b_{r+1}_', fit, f'node-{r+1} difficulties (logit)')

## A4. Trichotomous disengagement — GRM (probit) / GPCM (logit)

**Simulate** (data-generating step — separate from the package)

In [ ]:
spec_tri = ts.disengagement_tree(binary=False)
rng = np.random.default_rng(SEED+3)
theta1, theta2 = rng.normal(0,1,P), rng.normal(0,1,P)
b1_true  = rng.normal(0.6, 0.7, I)                      # node-1 binary difficulty
thr_true = np.sort(rng.normal(0,1,(I,2)), axis=1)       # node-2: 2 ordered thresholds/item
truth = [dict(theta=theta1, b=b1_true), dict(theta=theta2, b=thr_true)]
Y_probit = h.simulate_irtree(spec_tri, truth, P, I, np.random.default_rng(SEED+3),
                             node_types=['binary','grm'],  node_links=['probit','probit'])
Y_logit  = h.simulate_irtree(spec_tri, truth, P, I, np.random.default_rng(SEED+3),
                             node_types=['binary','gpcm'], node_links=['logit','logit'])
resp_probit, resp_logit = responses_frame(Y_probit), responses_frame(Y_logit)
print('category counts (probit):', np.bincount(Y_probit.ravel())[1:])

**Probit** — ingest, node models, fit, compare

In [ ]:
data = ing.load_data(resp_probit, spec=spec_tri, categories_start=1)
node_models = [nm.ProbitRaschNode(), nm.GRMNode(n_outcomes=3)]   # node 2 cumulative GRM
fit = ft.irtree_fit(data, spec_tri, node_models, n_iter=N_ITER, n_chains=N_CHAINS, seed=SEED+3)
compare(b1_true, 'b_1_', fit, 'node-1 difficulty (probit)')
print('node-2 GRM threshold params:', [n for n in fit.results[0].param_names if n.startswith('b_2_')][:6])

**Logit** — the same study, logit link

In [ ]:
data = ing.load_data(resp_logit, spec=spec_tri, categories_start=1)
node_models = [nm.LogitRaschNode(), nm.GPCMNode(n_outcomes=3)]   # node 2 adjacent-category GPCM
fit = ft.irtree_fit(data, spec_tri, node_models, n_iter=N_ITER, n_chains=N_CHAINS, seed=SEED+3)
compare(b1_true, 'b_1_', fit, 'node-1 difficulty (logit)')
print('node-2 GPCM step params:', [n for n in fit.results[0].param_names if n.startswith('b_2_')][:6])